In [ ]:
import json

import gc_utils
import matplotlib.pyplot as plt
import numpy as np

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
# sim_dir = "/Users/z5114326/Documents/simulations/"
sim_dir = "/Volumes/Expansion/simulations/"

sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

In [ ]:
Rd_dict = {sim: None for sim in sim_lst}

for sim in sim_lst:
    fire_dir = sim_dir + sim + "/" + sim + "_res7100"

    part = gc_utils.open_snapshot(600, fire_dir, ["star"])

    pxyz = part["star"].prop("host.distance.principal")
    mass = part["star"]["mass"]

    x = pxyz[:, 0]
    y = pxyz[:, 1]
    z = pxyz[:, 2]

    R = np.sqrt(x**2 + y**2)

    msk = (np.abs(z) < 0.3) & (R > 6.0) & (R < 12.0)

    Rsel = R[msk]
    msel = mass[msk]

    rbins = np.linspace(6, 12, 25)

    mass_in_bin, _ = np.histogram(Rsel, bins=rbins, weights=msel)

    area = np.pi * (rbins[1:] ** 2 - rbins[:-1] ** 2)

    Sigma = mass_in_bin / area

    rmid = 0.5 * (rbins[:-1] + rbins[1:])

    good = Sigma > 0

    coeff = np.polyfit(rmid[good], np.log(Sigma[good]), 1)

    Rd = -1.0 / coeff[0]
    Sigma0 = np.exp(coeff[1])

    Rd_dict[sim] = np.round(Rd, 1)

In [ ]:
save_json = "../data/sim_data/Rd_dict.json"

with open(save_json, "w") as f:
    json.dump(Rd_dict, f, indent=4)